*Import Library*

In [1]:
import gymnasium as gym
import numpy as np
import metaworld
import random
import os
os.environ['MUJOCO_GL']='osmesa'
import gymnasium as gym
from gymnasium.wrappers import ResizeObservation
from PIL import Image
import matplotlib.pyplot as plt
import importlib
import imageio
import time
from IPython.display import HTML
from base64 import b64encode
from metaworld.policies import *

In [2]:
mt50 = metaworld.MT50(42)
print(mt50.train_classes)

OrderedDict([('assembly-v2', <class 'metaworld.envs.mujoco.sawyer_xyz.v2.sawyer_assembly_peg_v2.SawyerNutAssemblyEnvV2'>), ('basketball-v2', <class 'metaworld.envs.mujoco.sawyer_xyz.v2.sawyer_basketball_v2.SawyerBasketballEnvV2'>), ('bin-picking-v2', <class 'metaworld.envs.mujoco.sawyer_xyz.v2.sawyer_bin_picking_v2.SawyerBinPickingEnvV2'>), ('box-close-v2', <class 'metaworld.envs.mujoco.sawyer_xyz.v2.sawyer_box_close_v2.SawyerBoxCloseEnvV2'>), ('button-press-topdown-v2', <class 'metaworld.envs.mujoco.sawyer_xyz.v2.sawyer_button_press_topdown_v2.SawyerButtonPressTopdownEnvV2'>), ('button-press-topdown-wall-v2', <class 'metaworld.envs.mujoco.sawyer_xyz.v2.sawyer_button_press_topdown_wall_v2.SawyerButtonPressTopdownWallEnvV2'>), ('button-press-v2', <class 'metaworld.envs.mujoco.sawyer_xyz.v2.sawyer_button_press_v2.SawyerButtonPressEnvV2'>), ('button-press-wall-v2', <class 'metaworld.envs.mujoco.sawyer_xyz.v2.sawyer_button_press_wall_v2.SawyerButtonPressWallEnvV2'>), ('coffee-button-v2', <

In [3]:
# Helper function to get the policy for a given environment
def get_policy(env_name):
    # Map environment names to their corresponding policy classes
    env_to_policy = {
        'assembly-v2': SawyerAssemblyV2Policy,
        'basketball-v2': SawyerBasketballV2Policy,
        'bin-picking-v2': SawyerBinPickingV2Policy,
        'box-close-v2': SawyerBoxCloseV2Policy,
        'button-press-topdown-v2': SawyerButtonPressTopdownV2Policy,
        'button-press-topdown-wall-v2': SawyerButtonPressTopdownWallV2Policy,
        'button-press-v2': SawyerButtonPressV2Policy,
        'button-press-wall-v2': SawyerButtonPressWallV2Policy,
        'coffee-button-v2': SawyerCoffeeButtonV2Policy,
        'coffee-pull-v2': SawyerCoffeePullV2Policy,
        'coffee-push-v2': SawyerCoffeePushV2Policy,
        'dial-turn-v2': SawyerDialTurnV2Policy,
        'disassemble-v2': SawyerDisassembleV2Policy,
        'door-close-v2': SawyerDoorCloseV2Policy,
        'door-lock-v2': SawyerDoorLockV2Policy,
        'door-open-v2': SawyerDoorOpenV2Policy,
        'door-unlock-v2': SawyerDoorUnlockV2Policy,
        'drawer-close-v2': SawyerDrawerCloseV2Policy,
        'drawer-open-v2': SawyerDrawerOpenV2Policy,
        'faucet-close-v2': SawyerFaucetCloseV2Policy,
        'faucet-open-v2': SawyerFaucetOpenV2Policy,
        'hammer-v2': SawyerHammerV2Policy,
        'hand-insert-v2': SawyerHandInsertV2Policy,
        'handle-press-side-v2': SawyerHandlePressSideV2Policy,
        'handle-press-v2': SawyerHandlePressV2Policy,
        'handle-pull-side-v2': SawyerHandlePullSideV2Policy,
        'handle-pull-v2': SawyerHandlePullV2Policy,
        'lever-pull-v2': SawyerLeverPullV2Policy,
        'peg-insert-side-v2': SawyerPegInsertionSideV2Policy,
        'peg-unplug-side-v2': SawyerPegUnplugSideV2Policy,
        'pick-out-of-hole-v2': SawyerPickOutOfHoleV2Policy,
        'pick-place-v2': SawyerPickPlaceV2Policy,
        'pick-place-wall-v2': SawyerPickPlaceWallV2Policy,
        'plate-slide-v2': SawyerPlateSlideV2Policy,
        'plate-slide-back-v2': SawyerPlateSlideBackV2Policy,
        'plate-slide-back-side-v2': SawyerPlateSlideBackSideV2Policy,
        'plate-slide-side-v2': SawyerPlateSlideSideV2Policy,
        'push-back-v2': SawyerPushBackV2Policy,
        'push-v2': SawyerPushV2Policy,
        'push-wall-v2': SawyerPushWallV2Policy,
        'reach-v2': SawyerReachV2Policy,
        'reach-wall-v2': SawyerReachWallV2Policy,
        'shelf-place-v2': SawyerShelfPlaceV2Policy,
        'soccer-v2': SawyerSoccerV2Policy,
        'stick-pull-v2': SawyerStickPullV2Policy,
        'stick-push-v2': SawyerStickPushV2Policy,
        'sweep-into-v2': SawyerSweepIntoV2Policy,
        'sweep-v2': SawyerSweepV2Policy,
        'window-close-v2': SawyerWindowCloseV2Policy,
        'window-open-v2': SawyerWindowOpenV2Policy
    }
    
    if env_name not in env_to_policy:
        raise ValueError(f"No policy found for environment {env_name}")
    
    # Return the policy instance
    return env_to_policy[env_name]()

In [5]:
# Test per verificare l'importazione di tutte le policy
# Lista di tutti gli ambienti in MT50
mt50_env_names = list(mt50.train_classes.keys())
print(f"Totale ambienti in MT50: {len(mt50_env_names)}")
print(f"Lista degli ambienti: {mt50_env_names}")

# Test di importazione delle policy per tutti gli ambienti
successful = 0
failed = []

for env_name in mt50_env_names:
    try:
        policy = get_policy(env_name)
        successful += 1
        print(f"✓ Importata con successo la policy per {env_name}")
    except Exception as e:
        failed.append((env_name, str(e)))
        print(f"✗ Fallita l'importazione della policy per {env_name}: {str(e)}")

print(f"\nRiepilogo: {successful}/{len(mt50_env_names)} policy importate con successo")
if failed:
    print("Policy non importate:")
    for env_name, error in failed:
        print(f"- {env_name}: {error}")

Totale ambienti in MT50: 50
Lista degli ambienti: ['assembly-v2', 'basketball-v2', 'bin-picking-v2', 'box-close-v2', 'button-press-topdown-v2', 'button-press-topdown-wall-v2', 'button-press-v2', 'button-press-wall-v2', 'coffee-button-v2', 'coffee-pull-v2', 'coffee-push-v2', 'dial-turn-v2', 'disassemble-v2', 'door-close-v2', 'door-lock-v2', 'door-open-v2', 'door-unlock-v2', 'hand-insert-v2', 'drawer-close-v2', 'drawer-open-v2', 'faucet-open-v2', 'faucet-close-v2', 'hammer-v2', 'handle-press-side-v2', 'handle-press-v2', 'handle-pull-side-v2', 'handle-pull-v2', 'lever-pull-v2', 'pick-place-wall-v2', 'pick-out-of-hole-v2', 'pick-place-v2', 'plate-slide-v2', 'plate-slide-side-v2', 'plate-slide-back-v2', 'plate-slide-back-side-v2', 'peg-insert-side-v2', 'peg-unplug-side-v2', 'soccer-v2', 'stick-push-v2', 'stick-pull-v2', 'push-v2', 'push-wall-v2', 'push-back-v2', 'reach-v2', 'reach-wall-v2', 'shelf-place-v2', 'sweep-into-v2', 'sweep-v2', 'window-open-v2', 'window-close-v2']
✓ Importata con s

In [3]:
from metaworld_env import RandomizeInitialPositionWrapper

In [ ]:
# Example usage:
env_name = 'reach-v2'
mt10 = metaworld.MT10(42)
env = mt10.train_classes[env_name](render_mode="rgb_array", camera_name="corner")
env_task_indices = [i for i, task in enumerate(mt10.train_tasks) if task.env_name == env_name]
# print(mt10.train_tasks)
task = mt10.train_tasks[random.choice(env_task_indices)]
env.set_task(task)

# first randomize positions
env = RandomizeInitialPositionWrapper(env)

policy = get_policy(env_name)()

In [ ]:
class MakeGoalObservableWrapper(gym.Wrapper):
    """A wrapper that makes the goal position observable in the environment observation."""
    
    def __init__(self, env):
        super().__init__(env)
        # Get the actual SawyerXYZEnv instance
        if hasattr(self.env, 'env'):
            sawyer_env = self.env.env
        else:
            sawyer_env = self.env
            
        # Make goal observable by setting partially_observable to False
        sawyer_env._partially_observable = False
        
        # Update observation space to reflect that goals are now visible
        # This is important if your RL algorithm checks the observation space bounds
        obs_space = sawyer_env.observation_space
        
        # Update the goal portion of observation space (last 3 elements) if needed
        if sawyer_env.goal_space is not None:
            high = obs_space.high.copy()
            low = obs_space.low.copy()
            
            # Set the goal bounds (last 3 elements)
            high[-3:] = sawyer_env.goal_space.high
            low[-3:] = sawyer_env.goal_space.low
            
            # Create updated observation space
            self.observation_space = gym.spaces.Box(
                low=low, 
                high=high,
                dtype=obs_space.dtype
            )

In [ ]:
import imageio
# Set up the environment with both wrappers
env_name = 'push-v2'
env = mt10.train_classes[env_name](render_mode="rgb_array", camera_name="corner")
env_task_indices = [i for i, task in enumerate(mt10.train_tasks) if task.env_name == env_name]
task = mt10.train_tasks[random.choice(env_task_indices)]
env.set_task(task)

print(env.n_rewards)
# Apply wrappers
env = RandomizeInitialPositionWrapper(env)
# env = MakeGoalObservableWrapper(env)

# Create policy
policy = get_policy_class(env_name)()

# Run an episode with the expert policy and save frames
def run_expert_policy_episode(env, policy, max_steps=200):
    frames = []
    obs, _ = env.reset()
    done = False
    step = 0
    total_reward = 0
    
    while not done and step < max_steps:
        # Get action from the expert policy
        action = policy.get_action(obs)
        
        # Execute action
        obs, reward, terminated, truncated, info = env.step(action)
        done = terminated or truncated
        total_reward += reward
        
        # Render and save frame
        frame = env.render()
        # Flip the frame vertically to correct the orientation
        frame = np.flipud(frame)
        frames.append(frame)
        
        step += 1
    
    print(f"Episode finished after {step} steps. Total reward: {total_reward}")
    return frames

# Run episode and collect frames
print(f"Running expert policy for {env_name}...")
frames = run_expert_policy_episode(env, policy)

# Save video
video_path = f"expert_policy_{env_name}.mp4"
imageio.mimsave(video_path, frames, fps=30)
print(f"Video saved to {video_path}")



Running expert policy for push-v2...
Episode finished after 200 steps. Total reward: 1207.2815056033587
Video saved to expert_policy_push-v2.mp4
